# Notebook 1 - P2 Prompt Engineering com Groq

Técnicas de prompting: Zero-Shot, Few-Shot, Chain-of-Thought

## Setup

In [ ]:
!pip install groq

In [ ]:
from google.colab import userdata

# Lê as chaves de API do Secrets do Colab
# Se a chave não existir, retorna None e o provider correspondente ficará indisponível
def get_secret(key_name):
    try:
        return userdata.get(key_name)
    except Exception:
        print(f"[AVISO] Secret '{key_name}' não encontrado. Configure no painel de Secrets.")
        return None

GROQ_API_KEY    = get_secret("GROQ_API_KEY")

print("Chaves carregadas:")
print(f"  Groq   : {'ok' if GROQ_API_KEY    else 'não configurada'}")

Chaves carregadas:
  Groq   : ok


## Função base

In [ ]:
import groq

groq_client = groq.Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

In [ ]:
def gerar_resposta(prompt, model="llama-3.1-8b-instant", temperature=0.1):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=500,
    )
    return response.choices[0].message.content

## Prompt Simples e Estruturado

### O que é um prompt?

Um **prompt** é qualquer texto que você fornece ao modelo como entrada. A qualidade e estrutura do prompt influenciam diretamente a qualidade da resposta.

Neste milestone você vai comparar:
- **Prompt vago:** instrução genérica, sem contexto
- **Prompt estruturado:** com restrições explícitas de formato e profundidade
- **Role prompting:** atribuindo um papel ao modelo

In [ ]:
# Prompt vago
prompt_vago = "Explique o que é inteligência artificial"

resposta_vaga = gerar_resposta(prompt_vago)
print("[PROMPT VAGO]")
print(resposta_vaga)

[PROMPT VAGO]
A inteligência artificial (IA) é uma área da ciência da computação que visa criar sistemas capazes de realizar tarefas que normalmente são associadas à inteligência humana, como aprender, raciocinar, resolver problemas e tomar decisões. A IA é baseada em algoritmos e técnicas de processamento de informações que permitem que os sistemas computacionais sejam capazes de:

1. **Aprendizado**: Os sistemas de IA podem aprender com dados e melhorar suas habilidades e conhecimentos ao longo do tempo.
2. **Raciocínio**: Os sistemas de IA podem realizar raciocínio lógico e resolver problemas de forma eficiente.
3. **Reconhecimento de padrões**: Os sistemas de IA podem identificar padrões em dados e tomar decisões com base nesses padrões.
4. **Comunicação**: Os sistemas de IA podem se comunicar com humanos de forma eficaz e entender linguagens naturais.

A IA é dividida em várias subáreas, incluindo:

1. **Aprendizado de Máquina**: É uma subárea da IA que visa criar sistemas capazes

In [ ]:
# Prompt estruturado
prompt_estruturado = """
Explique inteligência artificial para um estudante do ensino médio.
Sua resposta deve:
- Ter exatamente 3 tópicos numerados
- Usar linguagem simples, sem jargões
- Incluir um exemplo cotidiano em cada tópico
- Ter no máximo 150 palavras no total
"""

resposta_estruturada = gerar_resposta(prompt_estruturado)
print("[PROMPT ESTRUTURADO]")
print(resposta_estruturada)

[PROMPT ESTRUTURADO]
A inteligência artificial (IA) é uma área da ciência que visa criar máquinas capazes de realizar tarefas que normalmente são feitas por seres humanos. Aqui estão três tópicos importantes sobre IA:

1. **Aprendizado de Máquina**: É o processo pelo qual as máquinas aprendem a realizar tarefas com base em dados e experiências. Por exemplo, um aplicativo de recomendação de música pode aprender suas preferências musicais com base nas músicas que você ouve e marca como favoritas.

2. **Processamento de Linguagem Natural**: É a capacidade das máquinas de entender e gerar texto ou fala como os seres humanos. Por exemplo, um assistente virtual como Siri ou Google Assistant pode entender suas perguntas e responder de forma inteligente.

3. **Robótica**: É a área da IA que visa criar robôs capazes de realizar tarefas físicas. Por exemplo, um robô de limpeza pode aprender a navegar por um ambiente e limpar superfícies com base em suas instruções.


In [ ]:
# Prompt com um papel
prompt_role = """
Role:
Você é uma professora universitária especialista em IA,
conhecida por explicações didáticas e uso frequente de
analogias do cotidiano. Seu público são estudantes de graduação
sem experiência técnica anterior.

Pergunta:
O que é inteligência artificial?
"""

resposta_role = gerar_resposta(prompt_role)

print("[ROLE PROMPTING - Professora especialista]")
print(resposta_role)

[ROLE PROMPTING - Professora especialista]
Bem-vindos, alunos! Hoje vamos explorar um tópico fascinante: a inteligência artificial (IA). Imagine que você está em uma cafeteria e pede um café. Você não precisa explicar para o barista como fazer o café, ele já sabe. Ele tem uma espécie de "conhecimento" incorporado que lhe permite fazer o café da forma certa.

A inteligência artificial é como um barista virtual. É um sistema que pode aprender, raciocinar e tomar decisões de forma autônoma, sem precisar de intervenção humana direta. Ela pode processar grandes quantidades de dados, identificar padrões e fazer previsões baseadas em esses padrões.

A IA é composta por três componentes principais:

1. **Processamento de linguagem natural**: é como se o barista virtual pudesse entender o que você está dizendo e responder de forma apropriada. Isso é possível graças à capacidade da IA de processar linguagem natural, como texto ou voz.
2. **Aprendizado de máquina**: é como se o barista virtual pu

### Milestone 1

Compare as 3 saídas do modelo e avalie o desempenho com base nos seguintes parâmetros: cumprimento da resposta, tom e se seguiu todas as instruções.

## Zero-Shot vs One-Shot vs Few-Shot

O artigo fundacional do GPT-3 (https://arxiv.org/pdf/2005.14165) identificou três modos de uso:

| Modo | Descrição | Quando usar |
|---|---|---|
| **Zero-Shot** | Só a instrução, sem exemplos | Tarefas simples, instruções claras |
| **One-Shot** | Instrução + 1 exemplo | Quando o formato é não óbvio |
| **Few-Shot** | Instrução + 2–5 exemplos | Tarefas ambíguas ou com padrão específico |

**Intuição:** os exemplos não treinam o modelo, eles demonstram o padrão esperado diretamente no contexto.

### Tarefa: Classificação de Sentimento

Vamos classificar reviews como **Positivo**, **Negativo** ou **Neutro**.

In [ ]:
# ------ Zero-Shot ------
# Apenas a instrução, sem exemplos

reviews_teste = [
    "O produto chegou rápido, mas veio com uma peça quebrada.",
    "Simplesmente incrível! Superou todas as minhas expectativas.",
    "É um produto comum. Faz o que promete, nada além."
]

def classificar_zero_shot(review):
    prompt = f"""
Classifique o sentimento do review abaixo.
Responda com UMA palavra: Positivo, Negativo ou Neutro.

Review: {review}
Sentimento:"""
    return gerar_resposta(prompt)


print("[ZERO-SHOT — Classificação de Sentimento]\n")
for review in reviews_teste:
    resultado = classificar_zero_shot(review)
    print(f"Review : {review}")
    print(f"Resultado: {resultado.strip()}\n")

[ZERO-SHOT — Classificação de Sentimento]

Review : O produto chegou rápido, mas veio com uma peça quebrada.
Resultado: Negativo

Review : Simplesmente incrível! Superou todas as minhas expectativas.
Resultado: Positivo.

Review : É um produto comum. Faz o que promete, nada além.
Resultado: Neutro



In [ ]:
# ------ Few-Shot ------
# Instrução + 4 exemplos com pares (entrada → saída)

def classificar_few_shot(review):
    prompt = f"""
Classifique o sentimento do review.
Responda com UMA palavra: Positivo, Negativo ou Neutro.

Review: "Adorei! Qualidade excelente e entrega rápida."
Sentimento: Positivo

Review: "Horrível. Parou de funcionar em dois dias."
Sentimento: Negativo

Review: "Produto ok. Atende o básico."
Sentimento: Neutro

Review: "Não recomendo. Péssima qualidade pelo preço."
Sentimento: Negativo

Review: "{review}"
Sentimento:"""
    return gerar_resposta(prompt, temperature=0.0)


print("[FEW-SHOT — Classificação de Sentimento]\n")
for review in reviews_teste:
    resultado = classificar_few_shot(review)
    print(f"Review : {review}")
    print(f"Resultado: {resultado.strip()}\n")

[FEW-SHOT — Classificação de Sentimento]

Review : O produto chegou rápido, mas veio com uma peça quebrada.
Resultado: Negativo

Review : Simplesmente incrível! Superou todas as minhas expectativas.
Resultado: Positivo

Review : É um produto comum. Faz o que promete, nada além.
Resultado: Neutro



In [ ]:
# ------ Tabela comparativa ------
print("\n📊 COMPARAÇÃO ZERO-SHOT vs FEW-SHOT\n")
print(f"{'Review':<50} {'Zero-Shot':^12} {'Few-Shot':^12}")
print("-" * 76)

for review in reviews_teste:
    zs = classificar_zero_shot(review).strip()
    fs = classificar_few_shot(review).strip()
    review_curto = review[:47] + "..." if len(review) > 47 else review
    print(f"{review_curto:<50} {zs:^12} {fs:^12}")


📊 COMPARAÇÃO ZERO-SHOT vs FEW-SHOT

Review                                              Zero-Shot     Few-Shot  
----------------------------------------------------------------------------
O produto chegou rápido, mas veio com uma peça ...   Negativo     Negativo  
Simplesmente incrível! Superou todas as minhas ...  Positivo.     Positivo  
É um produto comum. Faz o que promete, nada alé...   Neutro.       Neutro   


### Milestone 2

Para este caso, no zero-shot e no few-shot os resultados do modelo são similares. Tente mudar o texto de algum dos `reviews_teste` para que o modelo Zero-Shot se confunda e não consiga classificar corretamente mas o Few-Shot sim. Quais foram as mudanças que você fez?

## Temperatura e Controle da Geração

### Conceito

A **temperatura** controla a aleatoriedade na seleção de tokens:

| Temperatura | Comportamento | Ideal para |
|---|---|---|
| `0.0` | Determinístico (sempre o token mais provável) | Extração de dados, classificação, CoT |
| `0.3–0.5` | Focado, mas com alguma variação | Resumos, traduções, Q&A |
| `0.7–0.8` | Balanceado | Geração geral de texto |
| `0.9–1.0` | Alta criatividade, mais imprevisível | Brainstorming, escrita criativa |

In [ ]:
# Efeito da temperatura em tarefa criativa
prompt_criativo = "Crie um slogan para uma startup de tecnologia sustentável"

temperaturas = [0.0, 0.3, 0.7, 1.0]

print("[TEMPERATURA — Efeito em tarefa criativa]\n")
for temp in temperaturas:
    resultado = gerar_resposta(prompt_criativo, temperature=temp)
    print(f"  temp={temp}: {resultado.strip()}")
    print(5*"***************")

[TEMPERATURA — Efeito em tarefa criativa]

  temp=0.0: Aqui estão algumas opções de slogans para uma startup de tecnologia sustentável:

1. "Inovando para um futuro mais verde"
2. "Tecnologia para um planeta mais sustentável"
3. "Transformando a tecnologia, transformando o mundo"
4. "Sustentabilidade em ação, tecnologia em ação"
5. "Conectando pessoas, protegendo o planeta"
6. "A tecnologia que cuida do planeta, cuida de você"
7. "Inovando para um mundo mais justo e sustentável"
8. "Tecnologia para uma vida mais saudável e sustentável"
9. "Desenvolvendo soluções para um futuro mais sustentável"
10. "A tecnologia que protege o planeta, protege o seu futuro"

Espero que essas opções sejam úteis para sua startup de tecnologia sustentável!
***************************************************************************
  temp=0.3: Aqui estão algumas sugestões de slogans para uma startup de tecnologia sustentável:

1. "Inovando para um futuro verde"
2. "Tecnologia para um mundo mais sustentável"

---
## In-Context Learning Profundo

### Conceito

In-context learning não é só para classificação. O modelo consegue **inferir regras complexas** a partir dos exemplos, incluindo:

- Formato de saída
- Tom e estilo de escrita
- Raciocínio por analogia
- Transformações de dados

Isso acontece porque o modelo foi treinado em bilhões de textos que contêm padrões de "entrada → saída". Os exemplos no prompt ativam esses padrões.

In [ ]:
# Aprender um formato de saída via exemplos
# O modelo vai aprender a gerar fichas técnicas a partir de descrições,
# sem que a "estrutura da ficha" seja descrita explicitamente.

prompt_formato = """
Descrição: Laptop ultrafino com 16GB RAM, tela OLED de 14", bateria de 12h
Ficha:
  TIPO      : Computador portátil
  MEMÓRIA   : 16 GB
  TELA      : 14 pol. OLED
  AUTONOMIA : 12 horas
  DESTAQUE  : Design ultrafino

Descrição: Fone de ouvido bluetooth, cancelamento de ruído ativo, 30h de bateria
Ficha:
  TIPO      : Fone de ouvido
  CONECTIV. : Bluetooth
  AUTONOMIA : 30 horas
  DESTAQUE  : Cancelamento de ruído ativo

Descrição: Smartwatch com GPS, monitor cardíaco, resistente à água até 50m
Ficha:"""

resposta_formato = gerar_resposta(prompt_formato, temperature=0.0)
print("Aprendendo formato de ficha técnica")
print(resposta_formato)

Aprendendo formato de ficha técnica
Aqui estão as descrições e fichas dos produtos:

**Laptop Ultrafino**

Descrição: Este laptop ultrafino é perfeito para quem precisa de uma máquina portátil e poderosa. Com 16GB de RAM, você pode executar múltiplas tarefas ao mesmo tempo sem problemas. A tela OLED de 14" oferece uma experiência visual incrível, com cores vibrantes e contraste alto. Além disso, a bateria dura até 12 horas, garantindo que você possa trabalhar ou estudar sem interrupções.

Ficha:
- TIPO: Computador portátil
- MEMÓRIA: 16 GB
- TELA: 14 pol. OLED
- AUTONOMIA: 12 horas
- DESTAQUE: Design ultrafino

**Fone de Ouvir Bluetooth**

Descrição: Este fone de ouvido Bluetooth é ideal para quem gosta de música e precisa de uma experiência de som imersiva. Com cancelamento de ruído ativo, você pode se concentrar no seu conteúdo sem distrações. A bateria dura até 30 horas, permitindo que você ouça música por horas a fio.

Ficha:
- TIPO: Fone de ouvido
- CONECTIV.: Bluetooth
- AUTONOMI

### Milestone 3

Com base nos dois últimos exemplos de Temperatura e In-Context Learning, que aplicação direta você poderia implementar para alguma tarefa que possa ser automatizada no seu ambiente de trabalho ou dia a dia?